# Setup

In [5]:
import json
import os
from typing import Any, List

from dotenv import load_dotenv
from libs.messages import UserMessage, SystemMessage, ToolMessage  # Different message types
from libs.tooling import tool  # Tool decorator for creating AI tools
from libs.llm import LLM  # Our Language Model wrapper

# Load environment variables

In [6]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

# Agent Initialization

In [28]:
class Agent:
    def __init__(self,
                role: str ="Personal Assistant",
                instructions: str = "Help users with any question",
                model: str = "gpt-4o-mini",
                temperature: float = 0.0,
                tools: List[Any]= None):
        """Initialize the agent with its configuration and tools

        Args:
            role: The agent's role/persona
            instructions: Basic instructions for the agent
            model: The LLM model to use
            temperature: Creativity parameter (0.0 = more 'deterministic')
            tools: List of tools the agent can use
        """
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature
        self.tools = tools
        # Initialize the LLM with tools if provided
        self.llm = LLM(api_key=api_key,
                       base_url=base_url,
                       model=model,
                       temperature=temperature,
                       tools=tools
                       )


    def invoke(self, user_message: str):
        """Process a user message and return a response

        Args:
            user_message: The user's input message

        Returns:
            The agent's response after processing tools if needed
        """
        messages = [
            SystemMessage(content=f"You're an AI Agent and your role is {self.role}. Your instructions: {self.instructions}")
        ]
        # Add user message to conversation
        messages.append(UserMessage(content=user_message))
        # Get AI response and add to conversation
        ai_message = self.llm.invoke(messages)
        messages.append(ai_message)

        # Check if tools were required
        while ai_message.tool_calls:
            # Process each tool call
            for call in ai_message.tool_calls:
                # Access tool call data correctly
                function_name = call.function.name
                function_args = json.loads(call.function.arguments)
                tool_call_id = call.id
                # Find the matching tool
                tool = next((t for t in self.tools if t.name == function_name), None)
                if tool:
                    result = tool(**function_args)
                    messages.append(
                        ToolMessage(content=json.dumps(result),
                        tool_call_id=tool_call_id,
                        name=function_name
                        ))
            # Get final AI response after tool usage and add to conversation
            ai_message = self.llm.invoke(messages)
            messages.append(ai_message)

        for m in messages:
            print(m)
        return ai_message.content


In [29]:
agent = Agent(role = "Coding Assistant")
response = agent.invoke("What is Python? Be concise")
print(response)

content="You're an AI Agent and your role is Coding Assistant. Your instructions: Help users with any question" role='system'
content='What is Python? Be concise' role='user'
content='Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used for web development, data analysis, artificial intelligence, scientific computing, and automation, among other applications.' role='assistant' tool_calls=None
Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used for web development, data analysis, artificial intelligence, scientific computing, and automation, among other applications.


# Calculator Tool: Create a tool for performing calculations and test the agent's ability to use it.

In [14]:
@tool
def calculator(expression: str) -> float:
    """Evaluate a mathematical expression"""
    return eval(expression)

# Register the calculator tool with the agent and test it with a query that requires calculation.

In [30]:
math_agent = Agent(role = "Math Assistant", tools=[calculator])
response = math_agent.invoke("What is 23 * 45?")
print(response)

content="You're an AI Agent and your role is Math Assistant. Your instructions: Help users with any question" role='system'
content='What is 23 * 45?' role='user'
content=None role='assistant' tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_lFCpbUrfwpg8QJaCLoRAvU1L', function=Function(arguments='{"expression":"23 * 45"}', name='calculator'), type='function')]
content='1035' role='tool' tool_call_id='call_lFCpbUrfwpg8QJaCLoRAvU1L' name='calculator'
content='The result of \\( 23 \\times 45 \\) is 1035.' role='assistant' tool_calls=None
The result of \( 23 \times 45 \) is 1035.


In [31]:
# Test multiple tool usage
response = math_agent.invoke("If I multiply 3 by 5, what do I get? Then later add 7")
print(response)

content="You're an AI Agent and your role is Math Assistant. Your instructions: Help users with any question" role='system'
content='If I multiply 3 by 5, what do I get? Then later add 7' role='user'
content=None role='assistant' tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_C7cpoP3cWcInbUTonx24rYiz', function=Function(arguments='{"expression": "3 * 5"}', name='calculator'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_XPPvu3wQ2pfN6rt5i9ZLVf0M', function=Function(arguments='{"expression": "(3 * 5) + 7"}', name='calculator'), type='function')]
content='15' role='tool' tool_call_id='call_C7cpoP3cWcInbUTonx24rYiz' name='calculator'
content='22' role='tool' tool_call_id='call_XPPvu3wQ2pfN6rt5i9ZLVf0M' name='calculator'
content='If you multiply 3 by 5, you get 15. Then, if you add 7 to that, the result is 22.' role='assistant' tool_calls=None
If you multiply 3 by 5, you get 15. Then, if you add 7 to that, the result is 22.


# Game tool

In [34]:
@tool
def get_games(num_games:int=1, top:bool=True):
    """
    Returns the top or bottom N games with highest or lowest scores.
    args:
        num_games (int): Number of games to return (default is 1)
        top (bool): If True, return top games, otherwise return bottom (default is True)
    """
    data = [
        {"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98},
        {"Game": "Super Mario Odyssey", "Platform": "Switch", "Score": 97},
        {"Game": "Metroid Prime", "Platform": "GameCube", "Score": 97},
        {"Game": "Super Smash Bros. Brawl", "Platform": "Wii", "Score": 93},
        {"Game": "Mario Kart 8 Deluxe", "Platform": "Switch", "Score": 92},
        {"Game": "Fire Emblem: Awakening", "Platform": "3DS", "Score": 92},
        {"Game": "Donkey Kong Country Returns", "Platform": "Wii", "Score": 87},
        {"Game": "Luigi's Mansion 3", "Platform": "Switch", "Score": 86},
        {"Game": "Pikmin 3", "Platform": "Wii U", "Score": 85},
        {"Game": "Animal Crossing: New Leaf", "Platform": "3DS", "Score": 88}
    ]
    # Sort the games list by Score
    # If top is True, descending order
    sorted_games = sorted(data, key=lambda x: x['Score'], reverse=top)

    # Return the N games
    return sorted_games[:num_games]


In [35]:
# Create an agent with the multiple tools
data_analyst_agent = Agent(
    role="Game Stats Assistant",
    instructions="You can bring insights about a game dataset based on users questions",
    tools=[get_games]
)

In [36]:
response = data_analyst_agent.invoke("What's the best game in the dataset?")
print(response)

content="You're an AI Agent and your role is Game Stats Assistant. Your instructions: You can bring insights about a game dataset based on users questions" role='system'
content="What's the best game in the dataset?" role='user'
content=None role='assistant' tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_vV6wgMXUO1x5xZhMoABBx0Kd', function=Function(arguments='{"num_games":1,"top":true}', name='get_games'), type='function')]
content='[{"Game": "The Legend of Zelda: Breath of the Wild", "Platform": "Switch", "Score": 98}]' role='tool' tool_call_id='call_vV6wgMXUO1x5xZhMoABBx0Kd' name='get_games'
content='The best game in the dataset is **The Legend of Zelda: Breath of the Wild** for the Switch, with a score of **98**.' role='assistant' tool_calls=None
The best game in the dataset is **The Legend of Zelda: Breath of the Wild** for the Switch, with a score of **98**.


In [37]:
response = data_analyst_agent.invoke("What's the worst game in the dataset?")
print(response)

content="You're an AI Agent and your role is Game Stats Assistant. Your instructions: You can bring insights about a game dataset based on users questions" role='system'
content="What's the worst game in the dataset?" role='user'
content=None role='assistant' tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_PCD3tJi4VXHbS1wEgtgWIXQi', function=Function(arguments='{"num_games":1,"top":false}', name='get_games'), type='function')]
content='[{"Game": "Pikmin 3", "Platform": "Wii U", "Score": 85}]' role='tool' tool_call_id='call_PCD3tJi4VXHbS1wEgtgWIXQi' name='get_games'
content='The worst game in the dataset is "Pikmin 3" for the Wii U, with a score of 85.' role='assistant' tool_calls=None
The worst game in the dataset is "Pikmin 3" for the Wii U, with a score of 85.
